# JOM 2025 Steemit Reproduction (OnChainGov)

This notebook reproduces the estimation strategy of the JOM 2025 paper by Prof. Yulin Fang and coauthors on the incentive effects of **governance tokens vs. tradeable tokens** in the Steemit platform. It turns the paper's one-off pipeline into a living, re-runnable document powered by OnChainGov.

**Pipeline**: collect Steemit content/curation events → compute token-incentive metrics (creation / curation / novelty / ownership share) → build a user-period panel → PSM-DID with placebo & event-study diagnostics → research-ready exports + paper charts.

## 1. Setup

Install OnChainGov (if not already installed) and bootstrap the notebook path.

In [ ]:
%pip install -q -e ".." 2>/dev/null || true

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from onchaingov.reproductions import run_demo, run_reproduction, synthesize_steemit_data
from onchaingov.reproductions.jom2025_steemit import ReproductionConfig
from onchaingov.export import event_study_chart, placebo_distribution_chart

## 2. Estimation strategy

The paper asks whether rewarding content with a **governance token** (stake-weighted, non-tradeable-ish) versus a **tradeable token** changes user contribution incentives. The econometric workhorse is a **PSM-DID**:

1. **Propensity score matching** on baseline characteristics (age, prior curation activity) to construct a balanced control group;
2. **Difference-in-differences** on the matched sample with entity and time fixed effects;
3. **Placebo tests** (in-time and in-space) to validate parallel trends.

OnChainGov reproduces each step with its indicators library, panel builder, PSM-DID template, and placebo templates.

## 3. End-to-end demo on synthetic data

The demo synthesizes a user-period panel with a known treatment effect on `creation` and runs the full PSM-DID pipeline. This is a **sanity check** that the toolchain can recover the paper's data-generating process.

In [ ]:
cfg = ReproductionConfig(event_time=__import__("datetime").datetime(2024, 7, 1), n_units=300, n_periods=14)
panel = synthesize_steemit_data(cfg)
summary = run_reproduction(panel, cfg, n_placebo=100)
print(summary["result"].summary_text())
print(f"Matched sample: {summary['matched_units']} rows")

### 3.1 Diagnostics

The reproduction also computes an **event study** (dynamic effects + pre-trend test) and an **in-space placebo** (randomized treatment assignment). Both should show no significant pre-trends / zero-centered placebo estimates when the true effect is present.

In [ ]:
es = summary["event_study"]
print(f"Event study: {len(es.relative_periods)} relative periods, pre-trends p-value = {es.pre_trends_pvalue():.4f}")
for r, c, s in zip(es.relative_periods, es.coefficients, es.std_errors):
    print(f"  rel {r:+d}: {c:.4f} (se {s:.4f})")

pb = summary["placebo"]
print(f"In-space placebo: mean pseudo-ATT {pb.mean_placebo:.4f}, empirical p-value {pb.empirical_pvalue:.4f} (passed={pb.passed()})")

### 3.2 Paper charts

Generate the figures used in a typical empirical IS paper: event-study coefficients with confidence intervals and the placebo distribution against the true ATT.

In [ ]:
event_study_chart(
    es.relative_periods, es.coefficients, es.std_errors,
    out_path="../data/reproductions/jom2025/event_study.png",
    title="Dynamic treatment effects (synthetic demo)",
)
placebo_distribution_chart(
    pb.placebo_estimates,
    out_path="../data/reproductions/jom2025/placebo.png",
    true_att=summary["att"],
    title="Placebo distribution (synthetic demo)",
)
print("charts written to ../data/reproductions/jom2025/")

## 4. Real data path

To reproduce on **real Steemit data**, first collect events, then run the reproduction from the collected frames. The treatment group (governance-token holders) is supplied via a file of user ids (one per line).

In [ ]:
# 1) Collect (this hits the live Steemit condenser API)
# !onchaingov collect steemit --tag life --limit 100 --with-votes --out ../data/raw

# 2) Reproduce from the collected events
# from onchaingov.reproductions import run_from_steemit_events
# summary = run_from_steemit_events(
#     "../data/raw",
#     out_dir="../data/reproductions/jom2025_real",
#     treated_file="../data/treated_users.txt",
# )
# print(summary["att"], summary["att_pvalue"])

## 5. Interpretation

- The **ATT** is the causal estimate of the governance-token incentive on the outcome (e.g. creation reward).
- The **event study** validates parallel pre-trends; insignificant pre-treatment coefficients support the DID identifying assumption.
- The **placebo** distribution should center on zero; an empirical p-value above 0.05 indicates the true ATT is unlikely to be driven by random treatment assignment.

See `data/reproductions/jom2025/` for the exported panel, matched sample, propensity scores, summary, and figures.